# Project Initialization and Data Preparation

**Author:** Jason Shang ([jzhoushang@gmail.com](mailto:jzhoushang@gmail.com)) \
**Step:** 00 \
**Objective:** Initialize working directories, download host reference genomes, and retrieve accessions from the NCBI SRA for downstream processing.

In [ ]:
# Imports

import io
import os
import subprocess
import sys

import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('..'))
from src import constants

In [ ]:
def run_esearch(species_name: str) -> pd.DataFrame:
    query = f'\\"{species_name}\\"[Organism] AND WGS[Strategy] AND ILLUMINA[Platform] AND PAIRED[Layout]'
    cmd = f'esearch -db sra -query "{query}" | efetch -format runinfo'

    # print("$", cmd)
    result = subprocess.run(
        cmd, 
        shell=True, 
        capture_output=True, 
        text=True, 
        check=True
    )

    if not result.stdout:
        return pd.DataFrame()

    df = pd.read_csv(
        io.StringIO(result.stdout),
        dtype=constants.ESEARCH_DTYPES,
        usecols=constants.ESEARCH_DTYPES.keys()
    )
    
    return df

In [ ]:
# Initialize working directories

os.makedirs("../data/accessions", exist_ok=True)
os.makedirs("../data/cleaned", exist_ok=True)
os.makedirs("../data/host_genomes", exist_ok=True)
os.makedirs("../data/host_db", exist_ok=True)

In [ ]:
# Collect accession numbers

samples = pd.DataFrame()
for subject in constants.SUBJECTS:
    df = run_esearch(subject)
    print(f"found {len(df)} results for {subject}")
    df_sorted = df.sort_values(by="size_MB", ascending=False).head(3)
    samples = pd.concat([samples, df_sorted], ignore_index=True)
print(f"collected {len(samples)} samples across {len(constants.SUBJECTS)} species")

print("saving metadata to ../data/accessions/sra_metadata.csv")
samples.to_csv("../data/accessions/sra_metadata.csv", index=False)
print("saving accessions to ../data/accessions/accessions.txt")
samples["Run"].to_csv("../data/accessions/accessions.txt", index=False, header=False)

In [ ]:
# Download host references genomes

for subject in constants.RESTRICTED_SUBJECTS:
    subject_snakecase = subject.lower().replace(" ", "_")
    try:
        subprocess.run(
            ["../scripts/download_host_genome.sh", subject, subject_snakecase],
            capture_output=True,
            check=True
        )
        print("genome found and saved for", subject)
    except subprocess.CalledProcessError:
        print("no genome found for", subject)